# Italy day-ahead electricity demand: task briefing

**Goal:** beat (or honestly match) Terna's published day-ahead forecast, using only information available at forecast time, and quantify how uncertain each forecast is.

This notebook is the common reference for everyone. Read **Section 0** (rules) before starting your task. The code that implements the rules is `split_config.py`, in the repository root. Import it. Don't keep a private copy of the dates or the column names.

| Task | Topic | Notes |
|---|---|---|
| 1 | Baselines | same local hour last week, seasonal average, Terna forecast |
| 2 | Linear regression | calendar + lags, no weather |
| 3 | Gradient boosting (point forecast) | no weather, observed weather (upper bound), forecast weather |
| 4 | Prediction intervals | quantile regression + conformal prediction |
| Leader | Error analysis + final comparison | proposed 2020 lockdown, August holidays, summer heatwaves |

**Forecast cut-off (decided):** the forecast for day D is issued on day D-1 at **10:00 Europe/Rome**.

Timeline: final CSV by `[TO FILL: Wednesday evening]`. If you need more time, please say so early.


## 0. Ground rules (same for everyone)

1. **Target.** Italy total load, hourly, for the whole local calendar day D. Timestamps are stored in **UTC**. A target day is every hour whose Europe/Rome date is D: 24 hours normally, 23 on the spring-forward Sunday, 25 on the fall-back Sunday. Those short and long days stay in one piece inside a split.
2. **No leakage.** A feature is allowed only if it would be known at the cut-off (D-1, 10:00 Europe/Rome). Regulation 543/2013 requires the actual load of an hour no later than one hour after that hour ends. The shared code treats an hour as published two hours after it starts. That is a deadline, not a measured ENTSO-E delay.
   - Safe load features are **fixed UTC shifts of 48 h, 168 h and 336 h**. On a normal day the 48 h source is on local day D-2. On the fall-back Sunday and the day after, the 23:00 target looks back to 00:00 on D-1, which is still before the cut-off and published about eight hours earlier. The **24 h lag is not safe**: late hours of D would need hours of D-1 that have not happened yet. Don't use it.
   - These shifts are not the same local clock hour. For about seven days after each clock change, `shift(168)` lands one local hour away. The seasonal naive baseline is defined separately (Task 1) and does use the local clock.
   - Never use a rolling statistic that includes the hour being predicted, or any same-day total.
   - Anything fitted (means, scalers, encoders, hyperparameters, conformal widths) is fitted on training data only.
3. **Weather, two separate experiments. Don't mix the files.**
   - **Upper bound.** Observed weather is the ERA5 file `data/processed/weather_clean.csv`. Temperatures are `temp_milano_c`, `temp_torino_c`, `temp_bologna_c`, `temp_roma_c`, `temp_napoli_c`, `temp_palermo_c`. There is no single `temp_obs` column. Results that use this file are labelled **"upper bound"** in tables, plots and filenames. ERA5 at the target hour was not known at 10:00 on D-1.
   - **Realistic forecast.** Not in the repository yet. The contract, once Francesco has checked it, is Open-Meteo's **Previous Runs API**, variable `temperature_2m_previous_day2`, `models=gfs_seamless`, for the **same six cities**, columns `temp_fc_milano_c` and so on. `previous_day2` is a **fixed lead of about 48 hours before each valid hour**. It is not one GFS run issued at 10:00 for the whole of day D, and it is older than the 00 UTC run that is usually public by 10:00. That is conservative. It is leakage-safe only after the checks below. **Never `previous_day1`** (later hours of D would come from a run that was not public at 10:00). **Never the stitched Historical Forecast series**, and never the live forecast endpoint: where the archive has no model run, those series can silently return a reanalysis.
   - **Pending verification (Francesco).** Coverage of `temperature_2m_previous_day2` for all six cities over the training and validation hours is **not verified**. He needs to confirm, for a sample that includes a winter week, a summer week and both 2024 clock changes: the valid time, the model initialisation time, and that a missing run is left as NaN rather than filled with ERA5. A 48-hour lead is still before 10:00 on D-1 even after a 4–6 hour dissemination delay, but that is an argument, not a download. `FORECAST_COVERAGE_VERIFIED` in `split_config.py` stays `False` until then. If coverage starts after 2022, the forecast model is trained only on the covered local days, and the report says so.
4. **Split (chronological, local dates)**, imported from `split_config.py`, never hardcoded.
   - Train: local 2022-01-16 through 2023-12-31.
   - Validation: local 2024. This is the holdout used to compare models.
   - Test: local **2025, do not touch** until the leader says so. `final_test` raises unless the leader passes `confirm=True`.
   - The UTC year boundary is not the local midnight. `2023-12-31 23:00 UTC` is `2024-01-01 00:00` in Rome, so it is validation. `2024-12-31 23:00 UTC` is `2025-01-01 00:00` in Rome, so it is test. The last row of the file, `2025-12-31 23:00 UTC`, is `2026-01-01 00:00` in Rome and is in none of the three sets.
   - Training starts on 16 January rather than 15 January because the series begins at 01:00 local on 1 January. Local `2022-01-15 00:00` has no 336 h history. From the 16th, every training hour has it.
5. **Same rows, same dataset.** Everyone loads `data/processed/dataset_no_weather.csv` through `load_data()`. Columns are `timestamp_utc`, `actual_load_mw`, `terna_forecast_mw`, plus the calendar columns already merged from Tomas. Prediction files still use the column name `timestamp` (explicit UTC). The leader writes down the freeze date before anyone touches 2025: `[TO FILL: freeze date]`.
6. **Known holiday labels.** Use `is_holiday` from the CSV so everyone has the same flag. It is 1 on the first Sunday of November (2022-11-06, 2023-11-05, 2024-11-03, 2025-11-02). Those Sundays are not civil public holidays. Don't rebuild a second holiday column with `holidays.IT()`: the two definitions would make the models incomparable. Don't edit the CSV in your notebook.
7. **Preliminary results.** If a dataset check changes anything material we will create a new file and rerun. Scripts take the path from `split_config.py`.
8. **Phase 2 is proposed, not a confirmed deliverable.** If the team approves it, electricity for 2018–2020 is collected separately, a no-weather model is trained only on pre-lockdown data, and errors are scored on the 2020 lockdown. That reuses this protocol. It does not reuse a model trained on 2022–2025, and it does not change the main experiment. Don't download that history as part of Tasks 1–4.


## 1. Shared configuration

`split_config.py` is in the repository root. Paths are resolved from that file, so the notebook's working directory does not matter as long as Python can import it.

```python
from split_config import *
df = load_data()
train, val = split(df)          # 2025 is not returned
```

Don't copy dates, column names or lag lengths by hand. The cell below checks that the import and the local-day split match this file.


In [ ]:
from pathlib import Path
import sys

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "split_config.py").exists():
        sys.path.insert(0, str(candidate))
        break

from split_config import (
    DATA_PATH,
    TRAIN_LOCAL_START,
    VAL_LOCAL_START,
    TEST_LOCAL_START,
    SAFE_LAGS_HOURS,
    FORECAST_COVERAGE_VERIFIED,
    load_data,
    split,
    local_midnight,
)

df = load_data()
train, val = split(df)
print("data:", DATA_PATH.name)
print("train local days:", TRAIN_LOCAL_START, "->", "2023-12-31", "rows", len(train))
print("val local days:  ", VAL_LOCAL_START, "->", "2024-12-31", "rows", len(val))
print("test local year stays closed; it would start", TEST_LOCAL_START)
print("safe UTC lags (hours):", SAFE_LAGS_HOURS)
print("forecast weather verified:", FORECAST_COVERAGE_VERIFIED)
print("val local-day lengths:", local_midnight(val.index).value_counts().value_counts().sort_index().to_dict())


## 2. Output format and checks

Every model saves its **validation** predictions as a CSV with exactly these columns:

`timestamp, y_true, y_pred, model`

- `timestamp` is explicit UTC (`2024-01-01T00:00:00Z` or `+00:00`), one row per validation hour.
- `y_true` is `actual_load_mw` for that hour, unchanged.
- `y_pred` is finite. No blanks.
- `model` is a short unique name: `seasonal_naive`, `seasonal_avg`, `terna`, `linreg_noweather`, `lgbm_noweather`, `lgbm_obs_upper_bound`, `lgbm_fc_gfs`.
- Interval models also save `y_lo, y_hi, nominal_coverage`. Extra columns are fine; the first four must exist.
- One file may contain several models. `(model, timestamp)` is unique, and `y_true` agrees across models.

`check_predictions` enforces that against `validation_index(df)`. Two documented exceptions:

- `seasonal_naive` may omit the hour whose local clock time did not exist seven days earlier (in 2024: 7 April, 02:00). Report the row count.
- `terna` on the **2025 test** may omit `2024-12-31T23:00:00Z` (local 2025-01-01 00:00), the only hour with no Terna forecast. That hour is not in the 2024 validation set, so the validation Terna file is complete. On the test, score Terna on the remaining hours and print both sample sizes. Don't fill the hole with the actual load, and don't drop it from the other models.

Call `check_predictions(path, validation_index(df), reference_y=df["actual_load_mw"])` before sending anything.


In [ ]:
from split_config import check_predictions, metrics, validation_index

# predictions = check_predictions(
#     "predictions/linreg_noweather.csv",
#     validation_index(df),
#     reference_y=df["actual_load_mw"],
# )
# print(metrics(predictions["y_true"], predictions["y_pred"]))


---
## Task 1: Baselines

**Purpose:** set the bar before modelling. If a complex model can't beat these, there is a bug or the features are not useful.

**Models (three):**
1. **Seasonal naive:** load at the **same Europe/Rome clock time seven local days earlier** (`seasonal_naive`). This is not `shift(168)`. Where that clock time does not exist, omit the hour and say so. Where the source hour is the repeated fall-back hour, the shared function uses the earlier UTC instant.
2. **Seasonal average:** mean load by local hour-of-week (168 groups), **computed on training data only** (`seasonal_average_prediction`). Both hours of a fall-back day contribute to hour 02:00.
3. **Terna's published day-ahead forecast**, column `terna_forecast_mw`, taken as-is. It is a benchmark, never a feature. On 2024 validation it has no missing hours.

**Deliverables**
- One CSV per baseline in the format of Section 2 (`seasonal_naive`, `seasonal_avg`, `terna`).
- MAE and RMSE on validation, with `n`, and a one-line comment on where each baseline fails (hint: holidays and the week of Ferragosto).

**Watch out for:** calling the UTC 168 h lag the seasonal naive. That lag is a legal model feature (Tasks 2 and 3). It is a different object from this baseline for about a week after each clock change.


In [ ]:
# Starter for Task 1
# from split_config import *
# df = load_data()
# train, val = split(df)
#
# naive = seasonal_naive(df).loc[val.index]
# seasonal_avg = seasonal_average_prediction(train, val)
# terna = val["terna_forecast_mw"]
# y_true = val["actual_load_mw"]


---
## Task 2: Linear regression (no weather)

**Purpose:** linear reference model for everyone, and a check that calendar + lags carry most of the signal.

**Features**
- Calendar already on the file, in local time: `hour_local`, `day_of_week`, `month`, `is_weekend`, `is_holiday`, `is_summer`, `is_christmas_period`. Encode hour and day of week as **one-hot or sin/cos, not plain integers**.
- Use the CSV `is_holiday`, including its known extra November Sundays (Section 0). Don't fit a second holiday definition.
- Lags from `add_lags`: **168 h and 336 h** required, **48 h** optional (report with and without). These are UTC shifts.
- No weather, and no Terna forecast.

**Deliverables**
- CSV `linreg_noweather` in the Section 2 format, MAE and RMSE on validation.
- A short table of which holiday-related errors are largest (e.g. Ferragosto, Christmas, bridges).

**Watch out for:** fitting scalers or encoders on training only. After the shared split, the 336 h lag is present on every training, validation and test row; still drop a row if you build a lag that is NaN. Don't read 2025.


In [ ]:
# Starter for Task 2
# import numpy as np
# from sklearn.linear_model import Ridge
# train, val = split(add_lags(load_data()))
# X = pd.DataFrame(index=train.index)
# X["is_holiday"] = train["is_holiday"].astype(int)
# X["hour_sin"] = np.sin(2 * np.pi * train["hour_local"] / 24)
# X["hour_cos"] = np.cos(2 * np.pi * train["hour_local"] / 24)
# X["load_lag168h"] = train["actual_load_mw_lag168h"]
# X["load_lag336h"] = train["actual_load_mw_lag336h"]
# # ... day of week, month, optional 48 h lag; fit on train only


---
## Task 3: Gradient boosting, point forecast

**Purpose:** the main model. Use **LightGBM** (suggested in the brief); if you prefer XGBoost, say so and stick to one in the report.

**Three versions of the same model** (identical calendar and lag features, identical hyperparameter search):
1. **No weather:** same features as Task 2, plus anything calendar-based you find useful (bridge days, holiday eve). Still no Terna column.
2. **Observed weather, upper bound:** add the six `temp_*_c` columns from `weather_clean.csv`, joined on UTC time. Heating or cooling degrees may be derived from those temperatures. A rolling temperature mean may use only hours at or before the target hour, and the whole version is labelled **"upper bound"**. This is not the operational model.
3. **Forecast weather:** the same six cities from `temperature_2m_previous_day2` (`temp_fc_*_c`), once Francesco's check has passed. Train on the forecast values, not on ERA5, so the model learns the forecast error. If an hour has no archived run, leave it out of this version. Don't fill it from `weather_clean.csv`. Until `FORECAST_COVERAGE_VERIFIED` is true, this version is blocked.

**Deliverables**
- Three CSVs: `lgbm_noweather`, `lgbm_obs_upper_bound`, `lgbm_fc_gfs` (the third only after the coverage check).
- Table: MAE, RMSE and `n` for the versions that exist, against the three baselines.
- Feature importance for the forecast version, or for the no-weather version if the forecast file is not ready.

**Rules:** tune hyperparameters with the expanding **local-day** folds from `chronological_local_day_folds`, applied to the **training** index only. Then fix the hyperparameters. Don't tune on 2024: it is the holdout used to compare models. Cap tuning at about 2 days.

**Watch out for:** a rolling mean that reaches past the target hour, and for treating `previous_day2` as "the forecast we would have downloaded at 10:00". It is an older fixed-lead series. That is acceptable only as the conservative experiment described in Section 0.


In [ ]:
# Starter for Task 3
# import lightgbm as lgb
# from split_config import chronological_local_day_folds, rows_on_local_days
# folds = chronological_local_day_folds(train.index, n_splits=4)
# for train_days, fold_days in folds:
#     inner_train = rows_on_local_days(train, train_days)
#     inner_val = rows_on_local_days(train, fold_days)
# model = lgb.LGBMRegressor(n_estimators=..., learning_rate=..., num_leaves=...)
# model.fit(X_train, y_train)
# y_pred = model.predict(X_val)


---
## Task 4: Prediction intervals

**Purpose:** say how uncertain each forecast is, and check whether the stated coverage is honest where it matters (the hours the grid cares about most).

Build the intervals on the **no-weather** model, so this task is not blocked by the forecast-weather file. If you also wrap the upper-bound model, label those rows `upper bound`.

**Part A, quantile regression**
- LightGBM with `objective="quantile"` at alpha = 0.1 and 0.9 (80% interval) as the main case; add 0.05/0.95 if time allows.
- Fit on the training local days only. Metrics: **pinball loss** and **empirical coverage** (share of true values inside the interval), on 2024.

**Part B, conformal prediction**
- A simple valid choice: fit the point model on the earlier training days, calibrate the interval on the last block of **training** local days (for example the last three to six months of 2023), and evaluate coverage on 2024. The calibration rows are not reused as if they were unseen.
- A sequential method (EnbPI, or MAPIE's time-series regressor) is also acceptable over 2024 if each interval uses only residuals from hours that were already in the past. Don't use in-sample training residuals: they are too small.
- Don't choose the interval width, the method, or the nominal level by looking at 2024 coverage and then quoting that same coverage.
- The coverage check on **2025** is run once, by the leader, at the end.

**Coverage must be reported by**
- season (DJF, MAM, JJA, SON), using the local date,
- Italian holidays vs normal days (`is_holiday`, with the November-Sunday caveat),
- heatwave days vs the rest of summer.

Heatwaves are an analysis label, not a feature. `heatwave_local_dates` uses the six ERA5 temperatures, the 95th percentile of training June–September days, and spells of at least three days. Applied to the six ERA5 series over 2022–2025, that rule marks only 17–19 July 2023 and 21–25 August 2023. **2024 and 2025 contain no such spell.** Say that the validation heatwave slice is empty. Don't lower the percentile after seeing 2024.

**Deliverables**
- CSV with `timestamp, y_true, y_pred, model, y_lo, y_hi, nominal_coverage` for each method.
- A coverage table and one plot (interval around the forecast for a hot week you define from training, and for Ferragosto week).
- Short comment: where does coverage fail, and does the conformal correction move it?

**Watch out for:** `from mapie.regression import MapieRegressor` depends on the installed MAPIE version. Check the API before relying on it. A training-tail calibration is enough if that import fights you.


In [ ]:
# Starter for Task 4
# lo = lgb.LGBMRegressor(objective="quantile", alpha=0.1).fit(X_train, y_train)
# hi = lgb.LGBMRegressor(objective="quantile", alpha=0.9).fit(X_train, y_train)
# coverage = ((y_val >= lo.predict(X_val)) & (y_val <= hi.predict(X_val))).mean()
#
# from split_config import heatwave_local_dates, load_observed_weather
# heatwave_days = heatwave_local_dates(load_observed_weather(), train.index)


---
## Leader tasks (for reference)

- **Final comparison:** concatenate the prediction CSVs, run `check_predictions` on the shared validation index, and compute MAE and RMSE in one place. Where `seasonal_naive` omits its one undefined hour, compare the other models on the common hours and also report their full-validation scores.
- **Error analysis:** August holidays and the heatwave definition above. The 2020 lockdown extension is **proposed**: it needs 2018–2020 load, a model trained only before the lockdown, and it stays outside the 2022–2025 table.
- **Test 2025:** run **once**, only on the baselines and the models chosen before looking at 2025, after the dataset version is frozen. Call `final_test(df, confirm=True)`. Score Terna on 8,759 hours and the other models on 8,760, and name the excluded timestamp `2024-12-31T23:00:00Z`.

## Known limits to mention in the report

- ENTSO-E can revise published load and the day-ahead forecast. Our lags and the Terna column are the values in the current file, not a snapshot taken at 10:00 on D-1. The Terna vintage (original day-ahead file versus a later revision) is **not yet verified**.
- `is_holiday` includes four remembrance Sundays in November. They are already weekends.
- 2022 is an unusual demand year (energy crisis), and it is in the training set.
- The stated heatwave rule does not flag any day in 2024 or 2025.
- Forecast weather is a fixed 48-hour GFS lead, not the single run available at 10:00, and its 2022–2025 coverage is **not yet verified**. The realistic weather model waits for that check.
- The seasonal naive baseline follows the local clock. The 168 h model lag follows UTC.
